In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet('data/processed/fraud_data.parquet')

In [3]:
datetimes = df["TransactionDT"].sort_values()
num_rows = datetimes.shape[0]
test_ratio, val_ratio = 0.70, 0.15

train_cutoff = datetimes.iloc[int(num_rows * test_ratio)]
val_cutoff = datetimes.iloc[int(num_rows * (test_ratio + val_ratio))]

train_set = df[df["TransactionDT"] < train_cutoff]
val_set = df[(df["TransactionDT"] >= train_cutoff) & (df["TransactionDT"] < val_cutoff)]
test_set = df[df["TransactionDT"] >= val_cutoff]

train_rows = train_set.shape[0]
val_rows = val_set.shape[0]
test_rows = test_set.shape[0]

assert train_rows + val_rows + test_rows == num_rows
print(f"Train: {(train_rows/num_rows * 100.0):.2f}")
print(f"Val: {(val_rows/num_rows * 100.0):.2f}")
print(f"Test: {(test_rows/num_rows * 100.0):.2f}")

assert train_set["TransactionDT"].max() < val_set["TransactionDT"].min()
assert val_set["TransactionDT"].max() < test_set["TransactionDT"].min()


Train: 70.00
Val: 15.00
Test: 15.00


In [4]:
# Check stats
print(f"Train Set Fraud Avg: {train_set['isFraud'].mean()}")
print(f"Val Set Fraud Avg: {val_set['isFraud'].mean()}")
print(f"Test Set Fraud Avg: {test_set['isFraud'].mean()}")

Train Set Fraud Avg: 0.03516878014795175
Val Set Fraud Avg: 0.03434145019812375
Test Set Fraud Avg: 0.03480430340592226


In [13]:
# Track version of dataset we use by hashing it
from pathlib import Path
from get_dataset_hash import file_fingerprint

dataset_path = Path("data/processed/fraud_data.parquet")
dataset_hash = file_fingerprint(dataset_path)
print(dataset_hash)

769f41ac9129c575856a3f366f33303dcd8f356fa046c500b77892a89188b9d6


In [16]:
import json

data = {
    "dataset_sha256": dataset_hash, # so we know what data version to apply split to 
    "train_end_id": int(train_cutoff),
    "val_end_id": int(val_cutoff),
}

dataset_config_path = Path(f"configs/data/versions/dataset_{dataset_hash[-8:]}.json")
dataset_config_path.parent.mkdir(parents=True, exist_ok=True)

with open(dataset_config_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)